# 使用 tiktoken 计算 token 数量

[`tiktoken`](https://github.com/openai/tiktoken/blob/main/README.md)是OpenAI开发的一种BPE分词器。
核心作用：
把文本拆分成模型真正看到的「token」单元
提前计算文本的 token 数量，避免超出模型上下文限制
预估 API 调用费用（按 token 计费）

为什么要用？
避免超出上下文限制：比如 GLM-4 上下文是 128k，如果你的对话总 token 超过这个数，模型会报错。
控制成本：智谱的计费是按「输入 token + 输出 token」算的，提前算输入 token 能帮你预估费用。
调试长文本：比如你上传知识库、写长提示词时，能知道文本会不会被截断。

什么时候会有误差？
对中文生僻词、特殊符号的分词结果，和智谱官方的分词器会有细微差异
日常开发、成本预估、上下文长度控制场景，这个误差完全可以忽略。


给定一段文本字符串（例如，`"tiktoken is great!"`）和一种编码方式（例如，`"cl100k_base"`），分词器可以将文本字符串切分成一系列的token（例如，`["t", "ik", "token", " is", " great", "!"]`）。

将文本字符串切分成token非常有用，因为GPT模型看到的文本就是以token的形式呈现的。知道一段文本字符串中有多少个token可以告诉你（a）这个字符串是否对于文本模型来说太长了而无法处理，以及（b）一个OpenAI API调用的费用是多少（因为使用量是按照token计价的）。

## 编码方式

编码方式规定了如何将文本转换成token。不同的模型使用不同的编码方式。
不同模型用不同的编码方式，可以直接用 tiktoken.encoding_for_model() 自动获取，如下所示：
```python
encoding = tiktoken.encoding_for_model('glm-4.5-air')
```

`tiktoken`支持OpenAI模型使用的三种编码方式：

| 编码名称           | 适用模型                                       |
|-------------------------|-----------------------------------------------------|
| `cl100k_base`           | `gpt-4`, `gpt-3.5-turbo`, `text-embedding-ada-002`,智谱 GLM-4/3、Kimi、通义千问（绝大多数现代模型都兼容） |
| `p50k_base`             | Codex模型, 老版 text-davinci 系列 `text-davinci-002`, `text-davinci-003`|
| `r50k_base` (或 `gpt2`) | 像 `davinci` 这样的早期GPT-3模型                         |


注意，`p50k_base` 与 `r50k_base` 有很大的重叠，对于非代码应用，它们通常会产生相同的token。

## 不同语言的分词器库

对于 `cl100k_base` 和 `p50k_base` 编码方式：

- Python: [tiktoken](https://github.com/openai/tiktoken/blob/main/README.md)
- .NET / C#: [SharpToken](https://github.com/dmitry-brazhenko/SharpToken), [TiktokenSharp](https://github.com/aiqinxuancai/TiktokenSharp)
- Java: [jtokkit](https://github.com/knuddelsgmbh/jtokkit)

对于 `r50k_base` (`gpt2`) 编码方式，许多语言都提供了分词器。

- Python: [tiktoken](https://github.com/openai/tiktoken/blob/main/README.md) (或者另选 [GPT2TokenizerFast](https://huggingface.co/docs/transformers/model_doc/gpt2#transformers.GPT2TokenizerFast))
- JavaScript: [gpt-3-encoder](https://www.npmjs.com/package/gpt-3-encoder)
- .NET / C#: [GPT Tokenizer](https://github.com/dluc/openai-tools)
- Java: [gpt2-tokenizer-java](https://github.com/hyunwoongko/gpt2-tokenizer-java)
- PHP: [GPT-3-Encoder-PHP](https://github.com/CodeRevolutionPlugins/GPT-3-Encoder-PHP)

（OpenAI对第三方库不做任何背书或保证。）

## 如何进行通常的分词操作

在英语中，token的长度通常在一个字符到一个单词之间变化（例如，`"t"` 或 `" great"`），尽管在某些语言中，token可以比一个字符短或比一个单词长。空格通常与单词的开头一起分组（例如，`" is"` 而不是`"is "` 或 `" "`+`"is"`）。你可以快速在 [OpenAI分词器](https://beta.openai.com/tokenizer) 检查一段字符串如何被分词。


## 0. 安装 `tiktoken`

In [1]:
pip install tiktoken

Looking in indexes: https://mirrors.cloud.tencent.com/pypi/simple/
Note: you may need to restart the kernel to use updated packages.


## 1. Import `tiktoken`

In [2]:
import tiktoken

## 2. Load an encoding

使用`tiktoken.get_encoding()`按名称加载编码。

第一次运行时，它将需要互联网连接进行下载。后续运行不需要互联网连接。

In [3]:
encoding = tiktoken.get_encoding("cl100k_base")

使用`tiktoken.encoding_for_model()`函数可以自动加载给定模型名称的正确编码。
encoding = tiktoken.encoding_for_model("gpt-3.5-turbo")
encoding_for_model() 函数只支持 OpenAI 自家的模型（比如 gpt-3.5-turbo、gpt-4），不支持智谱的 glm-4.5-air。

In [5]:
# 测试分词
text = "Say this is a test"
tokens = encoding.encode(text)
print(f"Token 数量：{len(tokens)}")
print(f"分词结果：{tokens}")
print(f"还原文本：{encoding.decode(tokens)}")

Token 数量：5
分词结果：[46864, 420, 374, 264, 1296]
还原文本：Say this is a test


通过计算`.encode()`返回的列表的长度来统计token数量。

In [7]:
def num_tokens_from_string(string: str, encoding_name: str) -> int:
    """返回文本字符串中的Token数量"""
    encoding = tiktoken.get_encoding(encoding_name)
    num_tokens = len(encoding.encode(string))
    return num_tokens

In [8]:
num_tokens_from_string("Say this is a test", "cl100k_base")

5

## 4. Turn tokens into text with `encoding.decode()`
`.decode()`将一个token整数列表转换为字符串。
**注意：尽管`.decode()`可以应用于单个token，但对于不在 utf-8 边界上的token来说，解码可能会有损失或错误。**
对于单个token，`.decode_single_token_bytes()` 安全地将单个整数token转换为其表示的字节。

In [9]:
[encoding.decode_single_token_bytes(token) for token in [83, 1609, 5963, 374, 2294, 0]]


[b't', b'ik', b'token', b' is', b' great', b'!']

（在字符串前面的`b`表示这些字符串是字节字符串。）